# Vichy Minéral 89: ¿El marketing dice lo mismo que los estudios publicados?

Vichy es una de las pocas marcas cosméticas con estudios indexados en PubMed: instrumentación nombrada (Corneómetro CM825, Tewameter TM300, Chromameter CR400), grupos control split-face, p-values reportados y registro NCT.

Este notebook extrae los datos numéricos de los 5 estudios publicados y los compara directamente contra los claims del marketing oficial de Vichy.

> Ejecuta `fetch_data.py` primero para descargar los metadatos desde NCBI. Este notebook usa los valores hardcodeados del full text, que son los datos más precisos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'font.family': 'monospace',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': '#fafafa',
    'axes.facecolor': '#fafafa',
})

## Datos de los estudios PubMed

Valores extraídos directamente del full text de cada paper (acceso abierto via PMC).

| Paper | n | Población | Hidratación | TEWL | Eritema | Diseño |
|---|---|---|---|---|---|---|
| [PMC7547125](https://pmc.ncbi.nlm.nih.gov/articles/PMC7547125/) | 20 | Rosácea | +33% | -11% | -28% | Split-face |
| [PMC9843703](https://pmc.ncbi.nlm.nih.gov/articles/PMC9843703/) | ? | Rosácea + mascarilla | +28% | -9% | -22% | RCT |
| [PMC9928536](https://pmc.ncbi.nlm.nih.gov/articles/PMC9928536/) | 38 | Piel envejecida + tretinoína | +11.5% | — | -48% | Split-face |
| [PMID 33786979](https://pubmed.ncbi.nlm.nih.gov/33786979/) | 47 | Post-procedimiento | — | — | -27.6% | Open-label |
| [PMID 33538111](https://pubmed.ncbi.nlm.nih.gov/33538111/) | 51 | Post-láser | +30% | -10% | -25% | Baseline comparison |

In [ ]:
# Datos extraídos de los papers (full text, acceso abierto)
studies = [
    {'label': 'PMC7547125\nRosacea split-face\n(n=20)',      'hydration': 33.0, 'tewl': -11.0, 'erythema': -28.0, 'population': 'Rosacea'},
    {'label': 'PMC9843703\nRosacea RCT',                     'hydration': 28.0, 'tewl':  -9.0, 'erythema': -22.0, 'population': 'Rosacea'},
    {'label': 'PMC9928536\nAntienvejecimiento\n(n=38)',      'hydration': 11.5, 'tewl':   None, 'erythema': -48.1, 'population': 'Aging + tretinoin'},
    {'label': 'PMID 33786979\nPost-procedimiento\n(n=47)',   'hydration':  None, 'tewl':   None, 'erythema': -27.6, 'population': 'Post-procedure'},
    {'label': 'PMID 33538111\nPost-laser\n(n=51)',           'hydration': 30.0, 'tewl': -10.0, 'erythema': -25.0, 'population': 'Post-laser'},
]

# Claim de marketing Vichy USA
marketing_hydration_claim = 100  # '100% de signos de hidratación' — autoevaluación n=42-53

print('Estudios con dato de hidratación:')
for s in studies:
    if s['hydration']:
        print(f"  {s['label'][:30].replace(chr(10),' ')}: +{s['hydration']}% ({s['population']})")
print(f"\nClaim de marketing Vichy: '{marketing_hydration_claim}% de signos de hidratación'")
print(f"Método del claim: autoevaluación, n=42-53, población no especificada")

## 1. Hidratación: datos de PubMed vs claim de marketing

Los estudios publicados miden mejoras de hidratación del **+11.5% al +33%** con corneómetro. El marketing dice **"100% de signos de hidratación"** — basado en autoevaluación de 42-53 personas.

In [ ]:
hyd_studies = [s for s in studies if s['hydration']]
labels = [s['label'] for s in hyd_studies]
vals = [s['hydration'] for s in hyd_studies]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(range(len(labels)), vals, color='#2563eb', alpha=0.7, edgecolor='white')
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=8)
ax.axvline(marketing_hydration_claim, color='#dc2626', linestyle='--', linewidth=1.5,
           label=f'Marketing: "{marketing_hydration_claim}% hidratación" (autoevaluación, n=42-53)')

for i, v in enumerate(vals):
    ax.text(v + 0.5, i, f'+{v}%', va='center', fontsize=9)

ax.set_xlabel('% mejora hidratación (Corneómetro, datos instrumentales)')
ax.set_title('Hidratación real en estudios PubMed vs claim de marketing\n(todos los estudios publicados: rosácea o piel comprometida, no piel general)', fontsize=10)
ax.legend(fontsize=9)
ax.set_xlim(0, 115)
plt.tight_layout()
plt.show()

## 2. TEWL en contexto: ¿es el -11% un efecto grande o pequeño?

Vichy reporta -11% de reducción TEWL en el estudio de rosácea split-face. ¿Es relevante clínicamente? Lo comparamos con valores de oclusivos documentados en literatura independiente.

In [ ]:
# Benchmark TEWL vs literatura independiente (Fluhr 2006, Darlenski 2009, Proksch 2008)
benchmark_tewl = {
    'Vaselina (Fluhr 2006)':              (-50, -35),
    'Aceite mineral (Darlenski 2009)':    (-35, -20),
    'Ceramidas (Proksch 2008)':           (-25, -12),
    'Vichy M89 (PMC7547125, rosácea)':   (-11, -11),
    'Hidratante genérico (Darlenski 2009)': (-18, -8),
}

fig, ax = plt.subplots(figsize=(9, 4))
for i, (name, (lo, hi)) in enumerate(benchmark_tewl.items()):
    if lo == hi:
        ax.scatter([lo], [i], color='#2563eb', zorder=5, s=100, marker='D')
        ax.text(lo - 0.5, i + 0.28, ' Vichy M89\n (PubMed)', fontsize=8, color='#2563eb')
    else:
        ax.barh(i, hi - lo, left=lo, height=0.5,
                color='#94a3b8' if 'Vichy' not in name else '#2563eb', alpha=0.65)
        ax.text(lo - 0.5, i, f'{lo}%', va='center', ha='right', fontsize=8)
        ax.text(hi + 0.5, i, f'{hi}%', va='center', ha='left', fontsize=8)

ax.set_yticks(range(len(benchmark_tewl)))
ax.set_yticklabels(list(benchmark_tewl.keys()), fontsize=8.5)
ax.set_xlabel('% reducción TEWL (negativo = mejora barrera)')
ax.set_title('TEWL: Vichy M89 vs literatura independiente\n(-11% está en el rango bajo — la vaselina llega a -50%)', fontsize=10)
ax.set_xlim(-60, 5)
plt.tight_layout()
plt.show()

print('El -11% de TEWL es estadísticamente significativo (p<0.001) pero clínicamente modesto.')
print('La vaselina, sin ningún activo específico, reduce el TEWL en 35-50%.')
print('Esto sugiere que el efecto es principalmente oclusivo, no específico del ingrediente.')

## 3. Población estudiada vs claim "para toda piel"

Los 5 estudios publicados de Vichy M89 son casi todos en poblaciones específicas con barrera cutánea comprometida. La rosácea tiene TEWL elevado de base, lo que da más margen de mejora que la piel sana.

In [ ]:
populations = {'Rosacea': 2, 'Post-procedimiento\no post-laser': 2, 'Aging skin\n+ tretinoin': 1}

fig, ax = plt.subplots(figsize=(7, 4))
wedges, texts, autotexts = ax.pie(
    list(populations.values()),
    labels=list(populations.keys()),
    autopct='%1.0f%%',
    colors=['#ef4444', '#f59e0b', '#7c3aed'],
    textprops={'fontsize': 9},
    startangle=90,
)
ax.set_title('Población de los 5 estudios publicados de Vichy M89', fontsize=11)

ax.text(0, -1.6,
        'Ningún estudio publicado valida el claim\n'
        'en piel sana general — todos son en condiciones\n'
        'con barrera cutánea comprometida.',
        ha='center', fontsize=9, color='#dc2626',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#fee2e2', edgecolor='#dc2626', alpha=0.8))

plt.tight_layout()
plt.show()

## 4. Calidad metodológica por estudio

Vichy publica más que la mayoría, pero eso también permite ver dónde están las debilidades.

In [ ]:
criteria = [
    'Instrumentación\nnombrada',
    'Grupo control\nexplícito',
    'p-values\nreportados',
    'Registro NCT',
    'Sin financiación\nde marca',
    'Población\ngeneral sana',
]
study_labels = ['PMC\n7547125', 'PMC\n9843703', 'PMC\n9928536', 'PMID\n33786979', 'PMID\n33538111']

matrix = np.array([
    [1,   0.5, 1,   0,   1  ],  # Instrumentacion nombrada
    [1,   1,   1,   0,   0.5],  # Grupo control
    [1,   1,   1,   1,   1  ],  # p-values
    [0,   1,   0,   0,   0  ],  # Registro NCT
    [0,   0,   0,   0,   0  ],  # Sin financiacion de marca
    [0,   0,   0,   0,   0  ],  # Poblacion general sana
])

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(study_labels)))
ax.set_xticklabels(study_labels, fontsize=8.5)
ax.set_yticks(range(len(criteria)))
ax.set_yticklabels(criteria, fontsize=8.5)
ax.set_title('Calidad metodológica por estudio (verde=sí / rojo=no / amarillo=parcial)', fontsize=10)

for i in range(len(criteria)):
    for j in range(len(study_labels)):
        v = matrix[i, j]
        text = 'Si' if v == 1 else 'Parcial' if v == 0.5 else 'No'
        ax.text(j, i, text, ha='center', va='center', fontsize=7.5,
                color='white' if v == 0 else 'black')

plt.colorbar(im, ax=ax, shrink=0.7, label='0=No  0.5=Parcial  1=Si')
plt.tight_layout()
plt.show()

## Conclusión

Vichy M89 es el caso más honesto de los tres analizados (Olay, Clarins, Vichy): publica estudios reales en PubMed con instrumentación declarada, p-values y en un caso registro NCT.

Pero el gap marketing vs evidencia existe:

1. **"100% de signos de hidratación"** → basado en autoevaluación de 42-53 personas, no en los estudios instrumentales (+11-33%)
2. **Todos los estudios publicados son en piel comprometida** (rosácea, post-procedimiento, post-láser) — no en piel sana general
3. **TEWL -11%** es estadísticamente significativo pero modesto vs oclusivos estándar
4. **Conflicto de interés**: todos los estudios están financiados por Vichy/L'Oréal. No hay estudios independientes del producto

Conclusión: el producto probablemente funciona como hidratante. Pero los claims del marketing usan metodología más débil que la publicada en PubMed, y extrapolan resultados de poblaciones específicas a "toda piel".